# 07 — Explainable clustering of daily mobility patterns
 **RQ8:** Can we discover distinct daily rental patterns from the hourly distribution **without supplying the calendar type to the algorithm**?

 Each day is represented by the 24 hourly *shares* of its daily rentals. The `workingday` label is reserved for post-hoc explanation and does not enter K-means. This connects directly with the in-class topics *Clustering*, *Supervised vs Unsupervised Learning*, and *Explaining Clusters*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

hour=pd.read_csv('../data/processed/bike_hour_clean.csv')
day=pd.read_csv('../data/processed/bike_day_clean.csv')
hour['dteday']=pd.to_datetime(hour['dteday'])
day['dteday']=pd.to_datetime(day['dteday'])
daily=day.set_index('dteday')
profiles=hour.pivot_table(index='dteday',columns='hr',values='cnt',aggfunc='sum')
profiles=profiles.reindex(columns=range(24))
excluded=profiles.isna().any(axis=1)
complete=profiles.loc[~excluded].copy()
complete=complete.loc[complete.sum(axis=1)>0]

print(f'Complete days: {len(complete)} of {len(profiles)} '
      f'({len(complete)/len(profiles):.1%})')
print('Mean daily rentals, complete vs excluded days:')
print('  Complete:', round(daily.loc[complete.index,'cnt'].mean()))
print('  Excluded:', round(daily.loc[profiles.index[excluded],'cnt'].mean()))

# Each of the 24 features is the fraction of a day's rentals at that hour.
X=complete.div(complete.sum(axis=1),axis=0)
scores={}
for k in range(2,7):
    km=KMeans(n_clusters=k,n_init=10,random_state=42)
    lab=km.fit_predict(X)
    scores[k]=silhouette_score(X,lab)
best_k=max(scores,key=scores.get)
print('Silhouette scores:',{k:round(v,3) for k,v in scores.items()})
print('Best among the tested k values:',best_k)

km=KMeans(n_clusters=best_k,n_init=10,random_state=42)
labels=km.fit_predict(X)
# Seed robustness check: agreement after changing the algorithm initialization.
other_labels=KMeans(n_clusters=best_k,n_init=10,random_state=123).fit_predict(X)
print('Adjusted Rand index across two seeds:',
      round(adjusted_rand_score(labels,other_labels),3))

res=pd.DataFrame({'cluster':labels},index=X.index)
res['workingday']=daily.loc[res.index,'workingday']
composition=pd.crosstab(res['cluster'],res['workingday'],normalize='index')*100
composition=composition.reindex(columns=[0,1],fill_value=0)
composition.columns=['Non-working','Working']
print('Cluster sizes:');print(res['cluster'].value_counts().sort_index())
display(composition.round(1))

In [ ]:
# Post-hoc meaning: assign a readable label using the withheld day type.
labels_text={}
for cluster_id in range(best_k):
    mostly_working=composition.loc[cluster_id,'Working']>50
    labels_text[cluster_id]=('Mostly working-day' if mostly_working
                             else 'Mostly day-off')+f' pattern (cluster {cluster_id})'

centroids=pd.DataFrame(km.cluster_centers_,columns=X.columns)*100
fig,axes=plt.subplots(1,2,figsize=(14,5.1))
cluster_colors=['#7C3AED','#0D9488','#B45309','#DC2626','#64748B','#2563EB']
for cluster_id in centroids.index:
    axes[0].plot(centroids.columns,centroids.loc[cluster_id],linewidth=2.6,
                 color=cluster_colors[cluster_id],
                 label=labels_text[cluster_id])
axes[0].set(title='Average hourly share in each discovered cluster',
            xlabel='Hour of day',ylabel='Share of daily rentals (%)')
axes[0].set_xticks(range(0,24,3))
axes[0].grid(axis='y',alpha=.16)
axes[0].legend(frameon=False,fontsize=9)

composition[['Non-working','Working']].plot(kind='bar',stacked=True,
            color=['#D97706','#2563EB'],ax=axes[1],width=.65)
axes[1].set(title='Calendar labels withheld from K-means',
            xlabel='Discovered cluster',ylabel='Composition of days (%)')
axes[1].set_xticklabels([str(i) for i in composition.index],rotation=0)
axes[1].set_ylim(0,110)
axes[1].legend(frameon=False,fontsize=9)
axes[1].grid(axis='y',alpha=.16);axes[1].set_axisbelow(True)
for bars in axes[1].containers:
    axes[1].bar_label(bars,fmt='%.0f%%',label_type='center',fontsize=9)

fig.suptitle('Unsupervised profiles recover the workday vs day-off rhythm',
             fontsize=14,fontweight='bold',y=1.04)
fig.text(.01,-.04,f'K-means on 24 within-day percentages; {len(complete)} complete days / '
         f'{len(profiles)} total; k={best_k} selected by silhouette among 2–6. '
         'UCI / Capital Bikeshare.',fontsize=9,color='#555555')
fig.tight_layout()
Path('../output/figures').mkdir(parents=True,exist_ok=True)
fig.savefig('../output/figures/RQ8_cluster_explanation.png',dpi=200,bbox_inches='tight')
plt.show()

### Crucial limitations
 - Days with absent hourly slots are excluded (never filled with zero). This excludes a non-random subset with lower average demand, so conclusions apply to the **complete-day subset**, not necessarily all 731 days.
 - K-means always assigns each included day to a group. We validate with silhouette scores, initialization stability, and **post-hoc interpretability**.
 - Clusters are not "commuters" or "leisure users" identified directly: we interpret *daily demand patterns*; actual trip motivations remain unknown.
 - `workingday` is used only for interpretation, avoiding information leakage into unsupervised learning.